# 📊 Data Visualization with DuckDB and Plotly

This notebook teaches you how to visualize data from DuckDB. We'll start with simple charts and work our way up to a complete dashboard.

For more info about plotly, go to: https://plotly.com  

## Table of Contents
1. [Setup and Imports](#setup-and-imports)
2. [Loading Sample Data](#loading-sample-data)
3. [Level 1: Simple Charts with Plotly Express](#-level-1-simple-charts-with-plotly-express)
4. [Level 2: Advanced Charts](#-level-2-advanced-charts)
5. [Level 3: Professional Customizations](#-level-3-professional-customizations)
6. [Level 4: Executive Dashboard](#-level-4-executive-dashboard)
7. [Summary](#-summary)

## Libraries Used

| Library | Description |
|---------|-------------|
| **DuckDB** | Fast analytical database that can run SQL queries directly on files |
| **Pandas** | Data manipulation and analysis with DataFrames |
| **Plotly Express** | High-level API for interactive visualizations - simple and fast |
| **Plotly Graph Objects** | Low-level API for more complex, customizable visualizations |

### Why Plotly?
- 🖱️ **Interactive**: Zoom, pan, hover tooltips out-of-the-box
- 🎨 **Modern**: Beautiful default designs
- 📱 **Responsive**: Adapts to different screen sizes
- 🔧 **Flexible**: From simple to complex - everything is possible

## Setup and Imports

In [1]:
pip install plotly statsmodels

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Basic imports
import duckdb
import pandas as pd

# Plotly for visualizations
import plotly.express as px           # Simple, fast charts
import plotly.graph_objects as go     # Complex, customizable charts
from plotly.subplots import make_subplots  # Combine multiple charts

# Create DuckDB connection
conn = duckdb.connect()
print("✓ Setup completed!")

✓ Setup completed!


## Loading Sample Data

We'll use the AdventureWorks dataset, which contains various sales data.

In [3]:
# Load tables from CSV files
conn.execute("""
    CREATE TABLE products AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.Product.csv');
    CREATE TABLE product_categories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductCategory.csv');
    CREATE TABLE product_subcategories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductSubcategory.csv');
    CREATE TABLE sales_orders AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderHeader.csv');
    CREATE TABLE sales_details AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderDetail.csv');
    CREATE TABLE territories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesTerritory.csv');
""")
print("✓ All tables loaded!")

# Quick overview
conn.execute("SHOW TABLES").df()

✓ All tables loaded!


,name
0,product_categories
1,product_subcategories
2,products
3,sales_details
4,sales_orders
5,territories


---
# 🟢 Level 1: Simple Charts with Plotly Express

Plotly Express (`px`) is the easiest way to quickly create meaningful charts.

## 1.1 Bar Chart

Perfect for comparing categories.

In [4]:
# Query revenue per territory
df_territory = conn.execute("""
    SELECT 
        t.Name AS territory, 
        ROUND(SUM(so.TotalDue), 2) AS revenue
    FROM sales_orders so 
    JOIN territories t ON so.TerritoryID = t.TerritoryID
    GROUP BY t.Name 
    ORDER BY revenue DESC
""").df()

# Simple bar chart - just 1 line of code!
fig = px.bar(df_territory, x='territory', y='revenue', title='💰 Revenue by Territory')
fig.show()

## 1.2 Horizontal Bar Chart

More readable with long labels.

In [5]:
# Top 10 products
df_products = conn.execute("""
    SELECT 
        p.Name AS product, 
        ROUND(SUM(sd.LineTotal), 2) AS revenue
    FROM sales_details sd 
    JOIN products p ON sd.ProductID = p.ProductID
    GROUP BY p.Name 
    ORDER BY revenue DESC 
    LIMIT 10
""").df()

# Horizontal bar chart with orientation='h'
fig = px.bar(
    df_products, 
    x='revenue', 
    y='product', 
    orientation='h',
    title='🏆 Top 10 Products by Revenue',
    color='revenue',
    color_continuous_scale='Blues'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

## 1.3 Line Chart

Ideal for time series and trends.

In [6]:
# Monthly revenue
df_monthly = conn.execute("""
    SELECT 
        DATE_TRUNC('month', OrderDate)::DATE AS month,
        ROUND(SUM(TotalDue), 2) AS revenue,
        COUNT(*) AS orders
    FROM sales_orders 
    GROUP BY 1 
    ORDER BY 1
""").df()

# Simple line chart
fig = px.line(
    df_monthly, 
    x='month', 
    y='revenue', 
    title='📈 Monthly Revenue Trend',
    markers=True  # Show data points
)
fig.show()

## 1.4 Pie Chart

Great for showing proportions of a whole.

In [7]:
# Revenue by product category
df_categories = conn.execute("""
    SELECT 
        pc.Name AS category, 
        ROUND(SUM(sd.LineTotal), 2) AS revenue
    FROM sales_details sd
    JOIN products p ON sd.ProductID = p.ProductID
    JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID
    GROUP BY pc.Name
""").df()

# Pie chart
fig = px.pie(
    df_categories, 
    values='revenue', 
    names='category', 
    title='🍰 Revenue Distribution by Category',
    hole=0.3  # Donut style (0 = regular pie)
)
fig.show()

## 1.5 Scatter Plot

Perfect for analyzing relationships between two variables.

In [8]:
# Orders with value and freight
df_orders = conn.execute("""
    SELECT 
        SubTotal,
        Freight,
        TotalDue
    FROM sales_orders
    WHERE SubTotal < 10000  -- Exclude outliers
    LIMIT 500
""").df()

# Scatter plot
fig = px.scatter(
    df_orders, 
    x='SubTotal', 
    y='Freight',
    title='📦 Order Value vs. Freight Costs',
    opacity=0.6,
    trendline='ols'  # Add trendline
)
fig.show()

---
# 🟡 Level 2: Advanced Charts

Now let's add more details and customizations.

## 2.1 Grouped Bar Chart

In [9]:
# Revenue by region and year
df_region_year = conn.execute("""
    SELECT 
        t."Group" AS region,
        YEAR(so.OrderDate) AS year,
        ROUND(SUM(so.TotalDue), 2) AS revenue
    FROM sales_orders so 
    JOIN territories t ON so.TerritoryID = t.TerritoryID
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()
df_region_year['year'] = df_region_year['year'].astype(str)

# Grouped bar chart
fig = px.bar(
    df_region_year,
    x='region',
    y='revenue',
    color='year',
    barmode='group',
    title='🌍 Revenue by Region and Year'
)
fig.show()

## 2.2 Stacked Area Chart

In [10]:
# Monthly revenue by category
df_monthly_cat = conn.execute("""
    SELECT 
        DATE_TRUNC('month', so.OrderDate)::DATE AS month,
        pc.Name AS category,
        ROUND(SUM(sd.LineTotal), 2) AS revenue
    FROM sales_orders so
    JOIN sales_details sd ON so.SalesOrderID = sd.SalesOrderID
    JOIN products p ON sd.ProductID = p.ProductID
    JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID
    GROUP BY 1, 2
    ORDER BY 1
""").df()

# Stacked area chart
fig = px.area(
    df_monthly_cat,
    x='month',
    y='revenue',
    color='category',
    title='📊 Revenue Development by Category'
)
fig.show()

## 2.3 Histogram

In [11]:
# Distribution of order values
df_order_values = conn.execute("""
    SELECT TotalDue
    FROM sales_orders
    WHERE TotalDue < 5000
""").df()

# Histogram
fig = px.histogram(
    df_order_values,
    x='TotalDue',
    nbins=50,
    title='📉 Distribution of Order Values',
    labels={'TotalDue': 'Order Value ($)'}
)
fig.update_layout(bargap=0.1)
fig.show()

## 2.4 Box Plot

In [12]:
# Order values by day of week
df_weekday = conn.execute("""
    SELECT 
        CASE DAYOFWEEK(OrderDate)
            WHEN 0 THEN '1-Sunday'
            WHEN 1 THEN '2-Monday'
            WHEN 2 THEN '3-Tuesday'
            WHEN 3 THEN '4-Wednesday'
            WHEN 4 THEN '5-Thursday'
            WHEN 5 THEN '6-Friday'
            WHEN 6 THEN '7-Saturday'
        END AS weekday,
        TotalDue
    FROM sales_orders
    WHERE TotalDue < 5000
""").df()

# Box plot
fig = px.box(
    df_weekday,
    x='weekday',
    y='TotalDue',
    title='📦 Order Value Distribution by Day of Week',
    color='weekday'
)
fig.show()

## 2.5 Heatmap

In [13]:
# Revenue by month and year
df_heatmap = conn.execute("""
    SELECT 
        YEAR(OrderDate) AS year,
        MONTH(OrderDate) AS month,
        ROUND(SUM(TotalDue), 0) AS revenue
    FROM sales_orders
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

# Pivot for heatmap
df_pivot = df_heatmap.pivot(index='year', columns='month', values='revenue')

# Heatmap
fig = px.imshow(
    df_pivot,
    title='🌡️ Revenue Heatmap (Year × Month)',
    labels={'x': 'Month', 'y': 'Year', 'color': 'Revenue'},
    color_continuous_scale='RdYlGn'
)
fig.show()

---
# 🔴 Level 3: Professional Customizations

With `update_layout()` and `update_traces()` we can create professionally styled charts.

## 3.1 Customized Line Chart

In [14]:
# Prepare data
df_monthly = conn.execute("""
    SELECT 
        DATE_TRUNC('month', OrderDate)::DATE AS month,
        SUM(TotalDue) AS revenue
    FROM sales_orders 
    GROUP BY 1 
    ORDER BY 1
""").df()

# Professional line chart
fig = px.line(df_monthly, x='month', y='revenue')

# Extensive customizations
fig.update_layout(
    title={
        'text': '📈 Monthly Revenue Trend',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 24}
    },
    xaxis_title='Time Period',
    yaxis_title='Revenue ($)',
    template='plotly_white',
    hovermode='x unified',
    yaxis_tickformat='$,.0f'
)

fig.update_traces(
    line=dict(width=3, color='#2E86AB'),
    mode='lines+markers',
    marker=dict(size=8)
)

fig.show()

## 3.2 Multiple Lines in One Chart (Dual Y-Axis)

In [15]:
# Revenue and orders per month
df_dual = conn.execute("""
    SELECT 
        DATE_TRUNC('month', OrderDate)::DATE AS month,
        SUM(TotalDue) AS revenue,
        COUNT(*) AS orders
    FROM sales_orders 
    GROUP BY 1 
    ORDER BY 1
""").df()

# Two Y-axes with Graph Objects
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=df_dual['month'], y=df_dual['revenue'], name='Revenue', line=dict(color='#2E86AB', width=3)),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=df_dual['month'], y=df_dual['orders'], name='Orders', line=dict(color='#F18F01', width=3)),
    secondary_y=True
)

fig.update_layout(
    title='📊 Revenue and Orders Over Time',
    template='plotly_white',
    hovermode='x unified'
)
fig.update_yaxes(title_text='Revenue ($)', secondary_y=False, tickformat='$,.0f')
fig.update_yaxes(title_text='Number of Orders', secondary_y=True)

fig.show()

---
# 🏆 Level 4: Executive Dashboard

Now let's combine everything into a professional dashboard with multiple visualizations!

In [16]:
# ====== PREPARE DATA ======

# 1. Monthly Revenue
monthly_df = conn.execute("""
    SELECT 
        DATE_TRUNC('month', OrderDate)::DATE AS month, 
        SUM(TotalDue) AS revenue 
    FROM sales_orders 
    GROUP BY 1 
    ORDER BY 1
""").df()

# 2. Categories
category_df = conn.execute("""
    SELECT 
        pc.Name AS category, 
        SUM(sd.LineTotal) AS revenue
    FROM sales_details sd 
    JOIN products p ON sd.ProductID = p.ProductID
    JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID 
    GROUP BY 1
""").df()

# 3. Top Territories
territory_df = conn.execute("""
    SELECT 
        t.Name AS territory, 
        SUM(so.TotalDue) AS revenue 
    FROM sales_orders so 
    JOIN territories t ON so.TerritoryID = t.TerritoryID 
    GROUP BY 1 
    ORDER BY 2 DESC 
    LIMIT 6
""").df()

# 4. KPIs
kpis = conn.execute("""
    SELECT 
        ROUND(SUM(TotalDue) / 1000000, 2) AS total_revenue_mio,
        COUNT(*) AS total_orders,
        ROUND(AVG(TotalDue), 2) AS avg_order_value,
        COUNT(DISTINCT CustomerID) AS unique_customers
    FROM sales_orders
""").df().iloc[0]

# 5. Yearly Growth
yearly_df = conn.execute("""
    SELECT 
        YEAR(OrderDate) AS year,
        ROUND(SUM(TotalDue), 0) AS revenue
    FROM sales_orders
    GROUP BY 1
    ORDER BY 1
""").df()

print("✓ Data loaded!")

✓ Data loaded!


In [17]:
# ====== CREATE DASHBOARD ======

# Define color palette
colors = {
    'primary': '#2E86AB',
    'secondary': '#F18F01',
    'success': '#28A745',
    'accent': '#A23B72',
    'background': '#F8F9FA'
}

# Dashboard Layout: 3 rows × 2 columns
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        '📈 Monthly Revenue',
        '🍰 Revenue by Category',
        '🏆 Top Territories',
        '📊 Yearly Revenue',
        '📋 Key Performance Indicators',
        ''
    ),
    specs=[
        [{'type': 'scatter'}, {'type': 'pie'}],
        [{'type': 'bar'}, {'type': 'bar'}],
        [{'type': 'table', 'colspan': 2}, None]
    ],
    row_heights=[0.35, 0.35, 0.30],
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# 1. Monthly Revenue (Line)
fig.add_trace(
    go.Scatter(
        x=monthly_df['month'], 
        y=monthly_df['revenue'],
        mode='lines+markers',
        line=dict(color=colors['primary'], width=3),
        marker=dict(size=6),
        name='Revenue',
        fill='tozeroy',
        fillcolor='rgba(46, 134, 171, 0.2)'
    ),
    row=1, col=1
)

# 2. Categories (Donut)
fig.add_trace(
    go.Pie(
        labels=category_df['category'], 
        values=category_df['revenue'],
        hole=0.4,
        marker_colors=[colors['primary'], colors['secondary'], colors['success'], colors['accent']],
        textinfo='label+percent',
        textposition='outside'
    ),
    row=1, col=2
)

# 3. Top Territories (Horizontal Bars)
fig.add_trace(
    go.Bar(
        x=territory_df['revenue'],
        y=territory_df['territory'],
        orientation='h',
        marker_color=colors['primary'],
        name='Territory'
    ),
    row=2, col=1
)

# 4. Yearly Revenue (Bars)
fig.add_trace(
    go.Bar(
        x=yearly_df['year'].astype(str),
        y=yearly_df['revenue'],
        marker_color=[colors['secondary'] if i < len(yearly_df)-1 else colors['success'] for i in range(len(yearly_df))],
        text=yearly_df['revenue'].apply(lambda x: f'${x/1000000:.1f}M'),
        textposition='outside',
        name='Yearly Revenue'
    ),
    row=2, col=2
)

# 5. KPI Table
fig.add_trace(
    go.Table(
        header=dict(
            values=['<b>💰 Total Revenue</b>', '<b>📦 Orders</b>', '<b>📊 Avg Order Value</b>', '<b>👥 Customers</b>'],
            fill_color=colors['primary'],
            font=dict(color='white', size=14),
            align='center',
            height=35
        ),
        cells=dict(
            values=[
                [f"${kpis['total_revenue_mio']:.2f}M"],
                [f"{kpis['total_orders']:,}"],
                [f"${kpis['avg_order_value']:,.2f}"],
                [f"{kpis['unique_customers']:,}"]
            ],
            fill_color=colors['background'],
            font=dict(size=16, color=colors['primary']),
            align='center',
            height=40
        )
    ),
    row=3, col=1
)

# ====== CUSTOMIZE LAYOUT ======
fig.update_layout(
    title={
        'text': '🎯 AdventureWorks Executive Dashboard',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 28, 'color': colors['primary']}
    },
    height=900,
    showlegend=False,
    template='plotly_white',
    paper_bgcolor=colors['background'],
    margin=dict(t=100, b=50, l=60, r=60)
)

# Format axes
fig.update_yaxes(tickformat='$,.0f', row=1, col=1)
fig.update_xaxes(tickformat='$,.0f', row=2, col=1)
fig.update_yaxes(categoryorder='total ascending', row=2, col=1)

fig.show()

---
## 📝 Summary

### Simple Charts (Plotly Express)
```python
px.bar(df, x='col', y='value')        # Bar chart
px.line(df, x='date', y='value')      # Line chart
px.pie(df, values='val', names='cat') # Pie chart
px.scatter(df, x='x', y='y')          # Scatter plot
px.histogram(df, x='value')           # Histogram
px.box(df, x='cat', y='value')        # Box plot
px.area(df, x='date', y='value')      # Area chart
```

### Customizations
```python
fig.update_layout(title='...', template='plotly_white')
fig.update_traces(line=dict(width=3, color='blue'))
fig.update_xaxes(tickangle=45)
fig.update_yaxes(tickformat='$,.0f')
```

### Dashboards
```python
from plotly.subplots import make_subplots
fig = make_subplots(rows=2, cols=2, specs=[...])
fig.add_trace(go.Scatter(...), row=1, col=1)
```

### Useful Links
- 📚 [Plotly Express Documentation](https://plotly.com/python/plotly-express/)
- 🎨 [Plotly Templates](https://plotly.com/python/templates/)
- 📊 [Chart Types Gallery](https://plotly.com/python/)

In [18]:
# Cleanup
conn.close()
print("✓ Done! Have fun visualizing! 🎉")

✓ Done! Have fun visualizing! 🎉
